## Setup

In [ ]:
import os
import optuna
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import average_precision_score
from optuna.integration import XGBoostPruningCallback
import cupy as cp

from src.py_src.models import GatekeeperModel

In [ ]:
load_dotenv()

XRAY_SLIDED_PATH = os.path.join(os.getenv("SLIDED_PATH"), "xray_slided.parquet")

xray_slided_df = pd.read_parquet(XRAY_SLIDED_PATH)

target_class = 'target_class_in_24h'
target_flux = 'target_flux_in_24h'
metadata_cols = ['run_id', 'time']

features = [col for col in xray_slided_df.columns if col not in metadata_cols + [target_class, target_flux]]

# Isolamento conceitual das features (X) e do alvo bruto (y)
X = xray_slided_df[features]
y = xray_slided_df[target_class]

In [ ]:
xray_slided_df.head()

## Preparing Data (Block-Chronological Split)

In [ ]:
cols_to_keep = features + ['time', target_class, target_flux]
gatekeeper_pool = xray_slided_df[cols_to_keep].copy()

def block_chronological_split(df, time_col, target_col, flux_col, _train_years, _val_years, _test_years, purge_hours=24):
    """
    Divide os dados em blocos baseados em anos para distribuir as fases do Ciclo Solar em todos os conjuntos.
    Aplica um Purge Gap Simétrico na transição de blocos:
    Remove 'purge_hours' do final do bloco anterior (para isolar o Target que olha pro futuro)
    E 'purge_hours' do início do novo bloco (para isolar as Features que olham pro passado).
    """
    df = df.sort_values(time_col).reset_index(drop=True).copy()
    df['year'] = df[time_col].dt.year

    # Atribuição inicial aos blocos
    df['split'] = 'none'
    df.loc[df['year'].isin(_train_years), 'split'] = 'train'
    df.loc[df['year'].isin(_val_years), 'split'] = 'val'
    df.loc[df['year'].isin(_test_years), 'split'] = 'test'

    # Remove amostras descartadas
    df = df[df['split'] != 'none'].reset_index(drop=True)

    # Identifica pontos de transição de bloco na linha do tempo
    df['block_change'] = df['split'] != df['split'].shift(1)
    df.loc[0, 'block_change'] = False # A primeira linha não é uma transição

    drop_indices = set()
    change_indices = df[df['block_change']].index
    purge_td = pd.Timedelta(hours=purge_hours)

    for idx in change_indices:
        transition_time = df.loc[idx, time_col]
        # Purga Simétrica: 'purge_hours' para trás e 'purge_hours' para frente
        start_purge = transition_time - purge_td
        end_purge = transition_time + purge_td

        to_drop = df[(df[time_col] >= start_purge) & (df[time_col] < end_purge)].index
        drop_indices.update(to_drop)

    df_purged = df.drop(index=list(drop_indices)).copy()

    dict_ = {'x': {}, 'y': {}, 'flux': {}}
    cols_to_drop = [target_col, flux_col, time_col, 'year', 'split', 'block_change']

    for split_name in ['train', 'val', 'test']:
        split_df = df_purged[df_purged['split'] == split_name].copy()
        dict_['x'][split_name] = split_df.drop(columns=cols_to_drop, errors='ignore')
        dict_['y'][split_name] = split_df[target_col].apply(lambda lb: 1 if lb >= 3 else 0)
        dict_['flux'][split_name] = split_df[flux_col]

    return dict_

# TREINO: A maior parte do Ciclo Solar 24 (Ascensão, Mínimo e Máximo misturados)
train_years = [2010, 2011, 2013, 2014, 2015, 2016, 2018, 2019]

# VALIDAÇÃO: Amostragem representativa do Ciclo 24 para o Optuna
# Pegamos 2012 (subida/atividade alta) e 2017 (descida/atividade baixa)
# Isso garante que o Optuna calibre o Threshold de forma equilibrada.
val_years = [2012, 2017]

# TESTE: O Ciclo Solar 25 inteiro (Isolado e intocado)
# Isso permite replicar exatamente a análise do artigo:
# - Avaliar em 2020-2021 (Min)
# - Avaliar em 2022 (Evolving)
# - Avaliar em 2023-2024 (Max)
test_years = [2020, 2021, 2022, 2023, 2024]

data = block_chronological_split(
    df=gatekeeper_pool,
    time_col='time',
    target_col=target_class,
    flux_col=target_flux,
    _train_years=train_years,
    _val_years=val_years,
    _test_years=test_years,
    purge_hours=24
)

print(f"Tamanho do Treino: {len(data['x']['train'])} amostras")
print(f"Tamanho da Validação: {len(data['x']['val'])} amostras")
print(f"Tamanho do Teste: {len(data['x']['test'])} amostras")

In [ ]:
data['x']['train'].head().to_csv()

In [ ]:
data['y']['train'].head().to_csv()

## Discovery Model

In [ ]:
discovery_model = GatekeeperModel(
    params={
        'n_estimators': 300,
        'learning_rate': 0.05,
        'max_depth': 5,
        'n_jobs': -1,
        'random_state': 42
    }
)

In [ ]:
selected_features = discovery_model.discover_top_features(
    x=data['x']['train'],
    y=data['y']['train'],
    flux_values=data['flux']['train'],
    cumulative_threshold=0.95
)

In [ ]:
selected_features

## Hyperparameter Tuning (Optuna)

In [ ]:
print("Transferindo dados para a VRAM da GPU...")

# 1. Filtramos as features selecionadas AINDA NO PANDAS (na CPU). Forçar 'float32' economiza 50% de VRAM e acelera o XGBoost
X_train_filtered = data['x']['train'][selected_features].astype('float32')
X_val_filtered = data['x']['val'][selected_features].astype('float32')

# 2. Convertendo para Arrays do CuPy (Isso move fisicamente os dados para a GPU)
X_train_gpu = cp.array(X_train_filtered.values)
y_train_gpu = cp.array(data['y']['train'].values.astype('float32'))

X_val_gpu = cp.array(X_val_filtered.values)
y_val_gpu = cp.array(data['y']['val'].values.astype('float32'))

print("Transferência concluída. Dados alocados na GPU.")

In [ ]:
def objective(trial):
    # 1. Calcula a proporção exata de desbalanceamento no Treino. Ratio = amostras_negativas / amostras_positivas
    neg_count = (data['y']['train'] == 0).sum()
    pos_count = (data['y']['train'] == 1).sum()
    imbalance_ratio = neg_count / pos_count if pos_count > 0 else 1.0

    pruning_callback = XGBoostPruningCallback(trial, 'validation_0-aucpr')

    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'aucpr',
        'n_estimators': 1000,
        'random_state': 1502,
        'n_jobs': -1,
        'device': 'cuda',

        'early_stopping_rounds': 50,
        'callbacks': [pruning_callback],

        # Espaço de busca ancorado na realidade física dos dados
        'scale_pos_weight': trial.suggest_float("scale_pos_weight", imbalance_ratio * 0.5, imbalance_ratio * 2.0),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'gamma': trial.suggest_float('gamma', 0.1, 5.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 25)
    }

    # features_to_keep=None porque os tensores (X_train_gpu) já estão filtrados
    model = GatekeeperModel(params=params, features_to_keep=None)

    # O XGBoost consumirá a memória diretamente da GPU
    model.fit(
        x=X_train_gpu,
        y=y_train_gpu,
        eval_set=[(X_val_gpu, y_val_gpu)],
        verbose=False
    )

    # retorna um array do CuPy (na GPU). Realiza a inferência (O XGBoost pode retornar CuPy ou NumPy dependendo da versão do wrapper)
    y_pred_proba_raw = model.predict_proba(X_val_gpu)[:, 1]

    # # trazer a inferência de volta pra CPU. Garantia de conversão segura para NumPy (CPU) para o Scikit-Learn
    y_pred_proba_cpu = y_pred_proba_raw.get() if hasattr(y_pred_proba_raw, 'get') else y_pred_proba_raw

    # O Target 'y_val' continua sendo o original do Pandas, pois está na CPU
    pr_auc = average_precision_score(data['y']['val'], y_pred_proba_cpu)

    return pr_auc

In [ ]:
study = optuna.create_study(direction='maximize')
print("\nIniciando tuning...")
study.optimize(objective, n_trials=500)

print(f"\nBest Score (PR AUC): {study.best_value:.4f}")
best_params = study.best_params

best_params.update({
    'n_estimators': 1000, 'objective': 'binary:logistic',
    'eval_metric': 'aucpr', 'random_state': 1502,
    'n_jobs': -1, 'early_stopping_rounds': 50
})

In [ ]:
final_model = GatekeeperModel(params=study.best_params, features_to_keep=selected_features)
final_model.fit(
    x=data['x']['train'], y=data['y']['train']
)

## Threshold Tuning

In [ ]:
# Calibração feita exclusivamente sobre o conjunto de validação para evitar Data Leakage
fig = final_model.get_threshold_graph(data['x']['val'], data['y']['val'])
# display(fig)

In [ ]:
optimal_threshold = final_model.optimize_threshold(
    data['x']['val'],
    data['y']['val'],
    target_recall=0.95,
    beta=2.0
)

## Results

In [ ]:
# ==========================================
# 1. PRÉ-COMPUTAÇÃO DE VETORES (Faz a inferência apenas UMA vez)
# ==========================================
x_test_df = data['x']['test']
y_true = data['y']['test'].values.astype(int)
flux_test = data['flux']['test']

y_prob = final_model.predict_proba(x_test_df)[:, 1]
y_pred = (y_prob >= optimal_threshold).astype(int)

# Dedução da Persistência: Se houve flare nas últimas 24h, preveja Alerta(1)
soma_flares_passado = x_test_df['count_C_24h'] + x_test_df['count_M_24h'] + x_test_df['count_X_24h']
y_persistence = (soma_flares_passado > 0).astype(int).values

In [ ]:
print("--- RELATÓRIO DE CLASSIFICAÇÃO ---")
print(final_model.get_classification_report(y_true, y_pred, target_names=['No Flare', 'Flare']))

In [ ]:
print("\n--- MÉTRICAS ABRANGENTES ---")
comprehensive_df = final_model.get_comprehensive_metrics(y_true, y_pred, y_prob)
display(comprehensive_df)

In [ ]:
print("\n--- PR-F1 (SKILL SCORE RELATIVO) ---")
pr_f1_score = final_model.calculate_prss(y_true, y_pred, y_persistence)
print(f"PR-F1 Score: {pr_f1_score:.4f}")

In [ ]:
print("\n--- ANÁLISE AC/NC (ACTIVITY CHANGE) ---")
ac_nc_df = final_model.analyze_ac_nc_performance(y_true, y_pred, y_persistence)
display(ac_nc_df)

In [ ]:
print("\n--- DISTRIBUIÇÃO DE ERROS POR CLASSE SOLAR ---")
error_dist_df = final_model.analyze_error_distribution(y_true, y_pred, flux_test)
display(error_dist_df)

In [ ]:
print("\n--- ANÁLISE DE FLUXO (ZONAS) ---")
# Defina seus buffer limits baseados na física, se houver (ex: [1e-6, 1e-5])
fig_flux, summary_flux = final_model.analyze_flux_errors(y_true, y_pred, flux_test, buffer_limits=[1e-6, 1e-5])
display(summary_flux)
display(fig_flux)

## Features Importance

In [ ]:
features_importance = final_model.get_feature_importance()
features_importance

## Export

In [ ]:
SAVE_PATH = os.getenv('GLOBAL_XRAY_MODELS_PATH')
os.makedirs(SAVE_PATH, exist_ok=True)

final_model.save(os.path.join(SAVE_PATH, 'gatekeeper_v1.joblib'))

print(f"Modelo Gatekeeper exportado com sucesso para: {SAVE_PATH}")
print(f"Threshold otimizado embutido: {final_model.threshold:.4f}")
print(f"Total de features retidas: {len(final_model.features_to_keep)}")